In [4]:
%pip install duckdb pymysql pandas requests

import duckdb
import pymysql
import pandas as pd
import requests
from datetime import datetime


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import duckdb
import pandas as pd
from datetime import datetime, timedelta


def s3_connection():
    """
    Teste la connexion S3 avec différentes dates
    """
    conn = duckdb.connect()
    
    try:
        # Installation et chargement des extensions
        print("Installation des extensions DuckDB...")
        conn.sql("INSTALL httpfs;")
        conn.sql("LOAD httpfs;")
        
        # Configuration S3
        conn.sql("SET s3_region='us-east-1';")
        conn.sql("SET s3_url_style='path';")
        conn.sql("SET s3_access_key_id='';")
        conn.sql("SET s3_secret_access_key='';")
        
        # Test avec différentes dates
        return  conn
        
    except Exception as e:
        print(f"Erreur lors de la configuration : {e}")
        return conn

def query_paris_data(conn):

    """
    Exécute la requête pour récupérer les données de Paris, Ile de France
    """
    try:

        query = f"""
            SELECT
                fsq_place_id,
                name,
                latitude,
                longitude,
                address,
                fsq_category_ids,
                locality,
                region
            FROM read_parquet(
                's3://fsq-os-places-us-east-1/release/dt=2024-11-19/places/parquet/*.parquet',
                union_by_name=true
            )
            WHERE latitude IS NOT NULL 
              AND longitude IS NOT NULL 
              AND country = 'FR' 
              AND (locality = 'Paris' OR locality='PARIS' )
              AND region IS NOT NULL 
              AND address IS NOT NULL
 
        """
        
        print(f"\nExécution de la requête pour Paris ")
        results = conn.sql(query).fetchall()
        
        # Conversion en DataFrame
        columns = ['fsq_place_id', 'name', 'latitude', 'longitude', 'address', 'fsq_category_ids', 'locality', 'region']
        df = pd.DataFrame(results, columns=columns)
        df['region']= "Ile de France"
        
        print(f"{len(df)} résultats trouvés pour Paris")
        return df
        
    except Exception as e:
        print(f"Erreur lors de l'exécution de la requête : {e}")
        return None


# Test de connexion
conn = s3_connection()

if conn:
    print(f"\nConnexion réussie")
    
    df_paris = query_paris_data(conn)# Requête pour Paris
    
else:
    print("Impossible de se connecter au bucket S3")
    
    
conn.close()

Installation des extensions DuckDB...

Connexion réussie

Exécution de la requête pour Paris 
133297 résultats trouvés pour Paris


In [2]:

# Étapes S3 pour DuckDB
duckdb.sql("INSTALL httpfs;")
duckdb.sql("LOAD httpfs;")
duckdb.sql("SET s3_region='us-east-1';")
duckdb.sql("SET s3_url_style='path';")

# Lecture des données depuis S3 (colonnes filtrées)
query = """
    SELECT
        fsq_place_id,
        name,
        latitude,
        longitude,
        address,
        fsq_category_ids,
        locality,
        region
    FROM read_parquet(
        's3://fsq-os-places-us-east-1/release/dt=2024-11-19/places/parquet/*.parquet',
        union_by_name=true
    )
    WHERE latitude IS NOT NULL AND longitude IS NOT NULL AND country= 'FR' AND (locality = 'Paris' OR locality='PARIS' ) AND region IS NOT NULL AND address IS NOT NULL
"""
print("recuperation des poi")
results_fsq = duckdb.sql(query).fetchall()
print("poi S3 ok")


recuperation des poi
poi S3 ok


In [2]:
import duckdb
import pandas as pd
from datetime import datetime, timedelta
# Étapes S3 pour DuckDB
duckdb.sql("INSTALL httpfs;")
duckdb.sql("LOAD httpfs;")
duckdb.sql("SET s3_region='us-east-1';")
duckdb.sql("SET s3_url_style='path';")

# Lecture des données depuis S3 (colonnes filtrées)
query = """
    SELECT
        fsq_place_id,
        name,
        latitude,
        longitude,
        address,
        fsq_category_ids,
        locality,
        region
    FROM read_parquet(
        's3://fsq-os-places-us-east-1/release/dt=2025-09-09/deltas/parquet/',
        union_by_name=true
    )
    
"""
print("recuperation")
results_fsq = duckdb.sql(query).fetchall()
print("S3 ok")


recuperation


InvalidInputException: Invalid Input Error: File 's3://fsq-os-places-us-east-1/release/dt=2025-09-09/deltas/parquet/' too small to be a Parquet file